In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("ethereum_ingestion_EDA")\
    .master("local[*]")\
    .config("spark.driver.memory", "6g")\
    .config("spark.executor.memory", "10g")\
    .config("spark.sql.adaptive.enabled", "true")\
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")\
    .config("spark.sql.autoBroadcastJoinThreshold", "100mb")\
    .config("spark.sql.parquet.filterPushdown", "true")\
    .config("spark.driver.maxResultSize", "4g")\
    .getOrCreate()


26/05/03 18:54:43 WARN Utils: Your hostname, khanhdo-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.118.128 instead (on interface ens33)
26/05/03 18:54:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 18:54:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/03 18:54:57 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
import pyspark.sql.functions as F

## Xử lý file tokens:
Bỏ trùng address + bỏ token decimal > 18

In [3]:
# tokens.parquet
df_tokens = spark.read.parquet("/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/tokens.parquet")
df_tokens.printSchema()
df_tokens.show(5)
df_tokens.count()

root
 |-- address: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- total_supply: string (nullable = true)

+--------------------+------+-----+--------+--------------------+
|             address|symbol| name|decimals|        total_supply|
+--------------------+------+-----+--------+--------------------+
|0x4bfbff63d03d93a...|      |     |      18|10000000000000000...|
|0x3498eaf5ec296eb...|    NM|NenMo|       4|        880000000000|
|0x9d1a236f1de06ab...| SPEED|Speed|      18|24229371788186338...|
|0x15ff21740d9e52a...|  WARP| Warp|      18|22443493719574827...|
|0xd32e32780734440...|      |     |      18|10000000000000000...|
+--------------------+------+-----+--------+--------------------+
only showing top 5 rows


174014

In [4]:
# Dedup + lọc decimals null
df_tokens = df_tokens.dropDuplicates(["address"]).filter(F.col("decimals").isNotNull())

df_tokens.printSchema()
df_tokens.show(5)
df_tokens.count()

root
 |-- address: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- total_supply: string (nullable = true)



+--------------------+------+--------------------+--------+--------------------+
|             address|symbol|                name|decimals|        total_supply|
+--------------------+------+--------------------+--------+--------------------+
|0x0000000000b3f87...|  GST2|         Gastoken.io|       2|             1361386|
|0x000000630a383f8...|  weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x000000d2870857f...|  weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x0000b6ab44789eb...|   VSN|      Vision Network|      18|           250000000|
|0x00016fab0fa144c...|  last|lite adaptable sy...|       1|           420000000|
+--------------------+------+--------------------+--------+--------------------+
only showing top 5 rows


174008

In [5]:

df_tokens_processed = df_tokens.filter(F.col("decimals") <= 18)
df_tokens_processed.distinct().show()

+--------------------+-------+--------------------+--------+--------------------+
|             address| symbol|                name|decimals|        total_supply|
+--------------------+-------+--------------------+--------+--------------------+
|0x0000000000b3f87...|   GST2|         Gastoken.io|       2|             1361386|
|0x000000630a383f8...|   weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x000000d2870857f...|   weth|Wrapped Ether (Ga...|      18|43290037992203770...|
|0x0000b6ab44789eb...|    VSN|      Vision Network|      18|           250000000|
|0x00016fab0fa144c...|   last|lite adaptable sy...|       1|           420000000|
|0x00024378720f481...|    TSR|     TesraAiSuperNet|      18|10000000000000000...|
|0x0006abbe90dc7e6...|  IG-11| Mandalorian.Finance|      18|10000000000000000...|
|0x00075b94bbb96c5...| gnarco|              gnarco|       0|                1000|
|0x0014c8459a7c5c3...|    SOL|             Solarix|       8| 1000000000000000000|
|0x001634b3ba2c8

In [12]:
import pandas as pd
pdf = df_tokens_processed.toPandas()
pdf.to_csv("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_tokens.csv", index=False)

In [6]:
file_token_transfers_paths = [
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_1.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_2.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/token_transfers_3.parquet"
]
df_token_transfers = spark.read.parquet(*file_token_transfers_paths)
df_token_transfers.printSchema()

df_token_transfers.show(5)
df_token_transfers.count()

root
 |-- token_address: string (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value: string (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|0xae53d5bdc95e83e...|0x0d6ce8158640cc0...|0x66a9893cc07d91d...|10910858345584184...|0x4fe83669b4f5045...|2026-03-08 05:36:47|
|0xc4d300d8cd9bc7e...|0x50024fcc0a8ecba...|0xb36a972147fb25a...|  267400000000000000|0x39099747d837b83...|2026-03-08 05:56:59|
|0x522e0d2beaaa4c4...|0xf78234a409b2174...|0x819567e155597c6...|            50239643|0xd12fa7

105170669

In [7]:
# Cast the 'value' column to decimal(38, 0)
df_token_transfers = df_token_transfers.withColumn("value", F.expr("try_cast(value as decimal(38,0))"))
df_token_transfers.printSchema()
df_token_transfers.show(5)

root
 |-- token_address: string (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value: decimal(38,0) (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+
|0xae53d5bdc95e83e...|0x0d6ce8158640cc0...|0x66a9893cc07d91d...|10910858345584184...|0x4fe83669b4f5045...|2026-03-08 05:36:47|
|0xc4d300d8cd9bc7e...|0x50024fcc0a8ecba...|0xb36a972147fb25a...|  267400000000000000|0x39099747d837b83...|2026-03-08 05:56:59|
|0x522e0d2beaaa4c4...|0xf78234a409b2174...|0x819567e155597c6...|            50239643|0

### Join df_token_transfers vs df_tokens_processed

In [8]:
df_joined = df_token_transfers.join(
    df_tokens_processed,
    df_token_transfers["token_address"] == df_tokens_processed["address"], how = 'inner')
df_joined.show(20)
count = df_joined.count()
print(f"Số lượng bản ghi sau khi join: {count}")

+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------+--------------------+--------+--------------------+
|       token_address|        from_address|          to_address|               value|    transaction_hash|    block_timestamp|             address|symbol|                name|decimals|        total_supply|
+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+------+--------------------+--------+--------------------+
|0x514910771af9ca6...|0x000000000004444...|0x51c72848c68a965...|28686159013835349808|0x931070935db0365...|2026-03-06 17:14:59|0x514910771af9ca6...|  LINK|     ChainLink Token|      18|10000000000000000...|
|0x464ebe77c293e47...|0x58edf7828133433...|0xb1b2d032aa2f523...|51402845000000000...|0xffbd25ede9abcf1...|2026-03-04 15:48:59|0x464ebe77c293e47...|   KRL|               Kryll| 

Số lượng bản ghi sau khi join: 36343432


In [9]:
df_joined_processed = df_joined.withColumn(
    "adjusted_value",
    F.col("value") / (10 ** F.col("decimals"))
).select(
    "transaction_hash",
    "from_address",
    "to_address",
    "adjusted_value",
    "block_timestamp",
    "token_address",
    "name",
    "symbol",
    "decimals"
)
df_joined_processed.show(10)

+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+---------------+------+--------+
|    transaction_hash|        from_address|          to_address|    adjusted_value|    block_timestamp|       token_address|           name|symbol|decimals|
+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+---------------+------+--------+
|0x931070935db0365...|0x000000000004444...|0x51c72848c68a965...| 28.68615901383535|2026-03-06 17:14:59|0x514910771af9ca6...|ChainLink Token|  LINK|      18|
|0xffbd25ede9abcf1...|0x58edf7828133433...|0xb1b2d032aa2f523...| 5140.284500000001|2026-03-04 15:48:59|0x464ebe77c293e47...|          Kryll|   KRL|      18|
|0x567b35ec35aa2e2...|0x990636ecb3ff04d...|0xbde1c0d9e1eaf3e...|  244.356571942466|2026-03-02 15:34:23|0x5a98fcbea516cf0...| Lido DAO Token|   LDO|      18|
|0x36ab827073ef22b...|0x91d40e4818f4d4c...|0x0084dfd7202e5

In [13]:
df_token_transfers_filtered = df_joined_processed.select(
    "from_address",
    "to_address",
    "token_address",
    "adjusted_value",
)

In [14]:
df_token_transfers_filtered.write.mode("overwrite").parquet("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_token_transfers.parquet")

In [28]:
# Tao 2 luong token ra va vao
df_inflow = df_joined_processed.select(
    F.lower(F.col("to_address")).alias("user_address"),
    F.lower(F.col("token_address")).alias("token_address"),
    F.col("adjusted_value").cast("double").alias("value"),
    F.col("block_timestamp")
)

df_outflow = df_joined_processed.select(
    F.lower(F.col("from_address")).alias("user_address"),
    F.lower(F.col("token_address")).alias("token_address"),
    (F.col("adjusted_value").cast("double") * F.lit(-1.0)).alias("value"),
    F.col("block_timestamp")
)

df_inflow.show(5, truncate=False)
df_outflow.show(5, truncate=False)

+------------------------------------------+------------------------------------------+-----------------+-------------------+
|user_address                              |token_address                             |value            |block_timestamp    |
+------------------------------------------+------------------------------------------+-----------------+-------------------+
|0x51c72848c68a965f66fa7a88855f9f7784502a7f|0x514910771af9ca656af840dff83e8264ecf986ca|28.68615901383535|2026-03-06 17:14:59|
|0xb1b2d032aa2f52347fbcfd08e5c3cc55216e8404|0x464ebe77c293e473b48cfe96ddcf88fcf7bfdac0|5140.284500000001|2026-03-04 15:48:59|
|0xbde1c0d9e1eaf3e53d8ce62de6fd0cf7f7c63bab|0x5a98fcbea516cf06857215779fd812ca3bef1b32|244.356571942466 |2026-03-02 15:34:23|
|0x0084dfd7202e5f5c0c8be83503a492837ca3e95e|0xff56cc6b1e6ded347aa0b7676c85ab0b3d08b0fa|286627.549       |2026-03-09 16:40:11|
|0x0432b731d718267b68de3daabc9572c6481ca605|0x3597bfd533a99c9aa083587b074434e61eb0a258|1545954.0        |2026-03-09 17

### Tính balance bằng tổng giá trị giao dịch với (vào: value +, ra: value - ), tổng số lần giao dịch mỗi token và lần giao dịch gần nhất

In [29]:
df_user_token_flow = df_inflow.union(df_outflow)

df_portfolios = (
    df_user_token_flow
    .groupBy("user_address", "token_address")
    .agg(
        F.sum("value").alias("balance"),
        F.count("*").alias("tx_count"), # Số lượng giao dịch liên quan đến token này
        F.max("block_timestamp").alias("last_active")
    )
    .filter(F.col("balance") > 0)
    .filter(F.col("user_address").isNotNull())
)

df_portfolios.show(20, truncate=False)
print("Portfolio rows:", df_portfolios.count())

+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+
|user_address                              |token_address                             |balance               |tx_count|last_active        |
+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+
|0x7809151cfef645a14a52f5903de04cb9d2a0d14b|0x1fcdce58959f536621d76f5b7ffb955baa5a672f|2.9103830456733704E-11|6       |2026-03-12 08:53:23|
|0x06fd4ba7973a0d39a91734bbc35bc2bcaa99e3b0|0x0f5d2fb29fb7d3cfee444a200298f468908cc942|170193.99999999945    |431     |2026-04-01 06:56:35|
|0xf3a4f293106722b35ef22efaf1a6503456a68f2f|0xa0b73e1ff0b80914ab6fe0444e65848c4c34450b|496.91308392          |1       |2026-03-11 02:31:23|
|0xdb567a8324fc903faf0ff9be595db08a9e47ea73|0x8bbe1a2961b41340468d0548c2cd5b7dfa9b684c|0.96647               |1       |2026-03-01 23:26:59|
|0x21a31ee1afc51d94c

Portfolio rows: 1574577


## Gán nhãn is_contract để phân biệt contract vs EOA

In [31]:
contracts_paths = [
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/contracts_1.parquet",
    "/home/khanhdo/Documents/project/bigdata_mining/data_ingestion/contracts_2.parquet"
]

df_contracts = (
    spark.read.parquet(*contracts_paths)
    .select(F.lower(F.col("address")).alias("address"))
    .dropDuplicates(["address"])
)

df_portfolios_labeled = (
    df_portfolios.alias("p")
    .join(df_contracts.alias("c"), F.col("p.user_address") == F.col("c.address"), "left")
    .withColumn(
        "is_contract",
        F.when(F.col("c.address").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
    .select(
        F.col("p.user_address").alias("user_address"),
        F.col("p.token_address").alias("token_address"),
        F.col("p.balance").alias("balance"),
        F.col("p.tx_count").alias("tx_count"),
        F.col("p.last_active").alias("last_active"),
        F.col("is_contract")
    )
)

df_portfolios_labeled.show(20, truncate=False)
print("Total portfolios:", df_portfolios_labeled.count())
print("Contracts:", df_portfolios_labeled.filter(F.col("is_contract") == True).count())
print("EOA users:", df_portfolios_labeled.filter(F.col("is_contract") == False).count())

+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance               |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|0x3cabc518fd45e71322eeeb3905d5bd46ad140328|0x5a98fcbea516cf06857215779fd812ca3bef1b32|10001.52              |1       |2026-03-08 14:50:35|false      |
|0x3e2b6d56cc69be167343ef7e81f5d8e6be163bd5|0xb8c77482e45f1f44de1745f52c74426c631bdd52|2.916111646300004E-7  |2       |2026-03-06 15:22:23|false      |
|0x6cbb2add2a126e65cef029afca45576910a26b5f|0x1f573d6fb3f13d689ff844b4ce37794d79a7ff1c|0.9000000000000017    |27      |2026-03-30 03:47:59|false      |
|0x91d40e4818f4d4c57b4578d9eca6afc92ac8debe|0x744d70fdbe2ba4cf95131626614a1763df805b9e|1

Total portfolios: 1574577


Contracts: 45111


EOA users: 1529466


## Lấy danh sách user có tổng giao dịch < 5000,  > 5000 khả năng lớn là ví sàn

In [ ]:
# Danh sach user EOA co tong giao dich < 5000 
df_real_users = (
    df_portfolios_labeled
    .filter(F.col("is_contract") == False)
    .groupBy("user_address")
    .agg(F.sum("tx_count").alias("total_tx_count"))
    .filter(F.col("total_tx_count") < 5000)
)

df_real_user_portfolios = (
    df_portfolios_labeled
    .filter(F.col("is_contract") == False)
    .join(df_real_users.select("user_address"), on="user_address", how="inner")
)

df_real_user_portfolios.printSchema()
df_real_user_portfolios.show(20, truncate=False)
print("Rows:", df_real_user_portfolios.count())

root
 |-- user_address: string (nullable = true)
 |-- token_address: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- last_active: timestamp (nullable = true)
 |-- is_contract: boolean (nullable = false)



+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance               |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+----------------------+--------+-------------------+-----------+
|0x0000401d0446ed6c6140d3dd106b42a7845c51d8|0xdac17f958d2ee523a2206206994597c13d831ec7|9.0E-4                |2       |2026-03-01 11:24:35|false      |
|0x00020d627c1c182d82bd7dcc7925c295af2e02a7|0xdac17f958d2ee523a2206206994597c13d831ec7|14.51                 |1       |2026-03-05 05:55:35|false      |
|0x000782fb09547704ae75cdcdad67ac1b8a72e1ab|0x0cc4f8735dae4b1ab7682cb05aeb19115e693e98|2.0E-18               |1       |2026-03-10 03:10:11|false      |
|0x0012e0d64b172bfbd8f0231db8b42a32b574a6e9|0xdac17f958d2ee523a2206206994597c13d831ec7|0

Rows: 1528520


In [ ]:
output_dir = "/home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios"
df_real_user_portfolios.write.mode("overwrite").parquet(output_dir)
print(f"Saved Spark parquet dataset to: {output_dir}")

Saved Spark parquet dataset to: /home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios
